# Risk sharing — Day 1
Run **Runtime → Run all** in Colab. Read the numbered cells in order.

We compare two ways of sharing wealth between two agents. This is a **provisional practice example**, pending Parsa's specification and independent calculation; it is not yet an optimization solver.

Saved outputs shown here come from a checked run in ChatGPT's Python environment. Run all cells to produce your own results.


## 1. Import NumPy
NumPy lets us calculate with arrays: lists of numbers arranged in rows and columns.


In [1]:
import numpy as np

## 2. Set the inputs
Three midpoint states: **1/6, 1/2, 5/6**. Each has probability **1/3**.
`wealth` is the amount available; `ranks` locate states in the probability distribution. They are equal here only because our target wealth distribution is uniform on [0, 1].

Edit agreed model inputs here, then rerun all cells. The utility functions below assume exactly two agents in order: h1, h2.


In [2]:
TOLERANCE = 1e-12
number_of_states, number_of_agents = 3, 2
ranks = (np.arange(number_of_states, dtype=float) + 0.5) / number_of_states
wealth = ranks.copy()  # Uniform [0,1] ONLY: keep a separate array.
probabilities = np.full(number_of_states, 1.0 / number_of_states)
pareto_weights = np.ones(number_of_agents)

print("Wealth:", wealth)
print("Ranks:", ranks)
print("Probability sum:", probabilities.sum())

Wealth: [0.1666666667 0.5          0.8333333333]
Ranks: [0.1666666667 0.5          0.8333333333]
Probability sum: 1.0


## 3. Define two ways to share wealth
Rows are wealth states; columns are agents. Equal sharing gives each agent half. The second example gives agent 1 one third and agent 2 two thirds.


In [3]:
allocations = {
    "equal": np.column_stack((wealth / 2, wealth / 2)),
    "one_third_two_thirds": np.column_stack((wealth / 3, 2 * wealth / 3)),
}

for name, consumption in allocations.items():
    print(f"\n{name}")
    print("  Wealth     Agent 1    Agent 2")
    for amount, row in zip(wealth, consumption):
        print(f"  {amount:.6f}   {row[0]:.6f}   {row[1]:.6f}")


equal
  Wealth     Agent 1    Agent 2
  0.166667   0.083333   0.083333
  0.500000   0.250000   0.250000
  0.833333   0.416667   0.416667

one_third_two_thirds
  Wealth     Agent 1    Agent 2
  0.166667   0.055556   0.111111
  0.500000   0.166667   0.333333
  0.833333   0.277778   0.555556


## 4. Check the sharing rules
An allocation is **feasible** when nobody receives a negative amount, each row adds up to wealth, and neither agent's consumption falls as wealth increases. The budget residual measures unallocated or overallocated wealth; it should be zero within rounding tolerance.


In [4]:
def check_feasibility(consumption):
    consumption = np.asarray(consumption, dtype=float)
    if consumption.shape != (number_of_states, number_of_agents) or not np.isfinite(consumption).all():
        raise ValueError(f"Allocation must be finite with shape ({number_of_states}, {number_of_agents}).")
    budget = float(np.max(np.abs(consumption.sum(axis=1) - wealth)))
    minimum = float(consumption.min())
    increment = float(np.diff(consumption, axis=0).min())
    return {
        "maximum_absolute_budget_residual": budget,
        "minimum_consumption": minimum,
        "minimum_monotonicity_increment": increment,
        "nonnegativity_violation": max(0.0, -minimum),
        "monotonicity_violation": max(0.0, -increment),
        "feasible": bool(budget <= TOLERANCE and minimum >= -TOLERANCE and increment >= -TOLERANCE),
    }

## 5. Calculate welfare
Agent 1 uses `log(1 + sqrt(rank) + consumption)`; agent 2 uses `log(1 + rank**(2/3) + consumption)`.
`np.log1p(x)` calculates the natural logarithm of `1 + x`. Each utility is multiplied by its probability and Pareto weight, then all contributions are added.


In [5]:
def calculate_welfare(consumption):
    utility = np.column_stack((
        np.log1p(np.sqrt(ranks) + consumption[:, 0]),
        np.log1p(ranks ** (2.0 / 3.0) + consumption[:, 1]),
    ))
    contributions = probabilities[:, None] * pareto_weights[None, :] * utility
    return float(contributions.sum()), contributions

## 6. Run the checks and compare results
The code also checks welfare using scalar arithmetic. That software check does not replace Parsa's independent hand calculation.


In [6]:
import math

assert np.isfinite(wealth).all() and np.isfinite(ranks).all() and np.isfinite(probabilities).all()
assert np.all(wealth >= 0) and np.all(np.diff(wealth) > 0)
assert np.all((ranks > 0) & (ranks < 1)) and np.all(np.diff(ranks) > 0)
assert np.all(probabilities >= 0) and abs(float(probabilities.sum()) - 1.0) <= TOLERANCE
assert pareto_weights.shape == (number_of_agents,) and np.all(pareto_weights > 0)

assert number_of_states >= 2, "Use at least two wealth states."
assert number_of_agents == 2, "This example implements exactly two utility types."
assert wealth.shape == ranks.shape == probabilities.shape == (number_of_states,)

results = {}
for label, consumption in allocations.items():
    check = check_feasibility(consumption)
    assert check["feasible"], (label, check)
    value, terms = calculate_welfare(consumption)
    # Scalar arithmetic cross-check, NOT a substitute for Parsa's hand calculation.
    scalar_value = sum(
        float(probabilities[j]) * (
            float(pareto_weights[0]) * math.log(1 + math.sqrt(float(ranks[j])) + float(consumption[j, 0]))
            + float(pareto_weights[1]) * math.log(1 + float(ranks[j]) ** (2 / 3) + float(consumption[j, 1]))
        ) for j in range(number_of_states)
    )
    assert abs(value - scalar_value) <= TOLERANCE
    results[label] = {**check, "welfare": value,
                      "scalar_crosscheck_absolute_error": abs(value - scalar_value)}

print(f"{'Allocation':<25} {'Feasible':<10} {'Budget error':<14} Welfare")
for name, result in results.items():
    print(f"{name:<25} {str(result['feasible']):<10} "
          f"{result['maximum_absolute_budget_residual']:<14.2e} {result['welfare']:.12f}")

Allocation                Feasible   Budget error   Welfare
equal                     True       0.00e+00       1.236217807544
one_third_two_thirds      True       0.00e+00       1.235554019370


With the original inputs, welfare should be **1.236217807544** for equal sharing and **1.235554019370** for one third / two thirds. Both should be feasible.

Equal sharing scores slightly higher **among these two choices**; this does not establish the best possible allocation. These expected values apply only to the original inputs.


## 7. Check that errors are detected
These deliberate mistakes check that the program rejects an incorrect budget, negative consumption, and decreasing consumption. They do not change our two valid allocations.


In [7]:
bad_budget = allocations["equal"].copy()
bad_budget[1, 0] += 0.01
assert not check_feasibility(bad_budget)["feasible"]

bad_domain = allocations["equal"].copy()
bad_domain[0] = [-0.01, wealth[0] + 0.01]
assert not check_feasibility(bad_domain)["feasible"]

bad_monotone = allocations["equal"].copy()
bad_monotone[-2] = [wealth[-2], 0.0]
bad_monotone[-1] = [0.0, wealth[-1]]
assert not check_feasibility(bad_monotone)["feasible"]

print("PASS: input, feasibility, welfare, and error-detection checks.")
print("Parsa's specification and independent calculation: pending.")

PASS: input, feasibility, welfare, and error-detection checks.
Parsa's specification and independent calculation: pending.


## 8. Save results (support code, collapsed)
This runs with **Run all**. It quietly records software versions and saves the exact inputs, diagnostics, and per-state calculations in `risk_sharing_day1/`. Download `day1_results.zip` from Colab's Files panel and save your notebook separately. The archive contains results, not the notebook or source code.

Files are replaced on rerun. Nothing is uploaded to GitHub by this notebook. Expand the cell only if you want to inspect how saving works.


In [8]:
#@title Save results and environment record
import json
import platform
import sys
import zipfile
from pathlib import Path
from importlib.metadata import version, PackageNotFoundError

OUT = Path("risk_sharing_day1")
OUT.mkdir(exist_ok=True)

for label, consumption in allocations.items():
    value, terms = calculate_welfare(consumption)
    np.savetxt(
        OUT / (label + ".csv"),
        np.column_stack((wealth, ranks, probabilities, consumption, terms, terms.sum(axis=1))),
        delimiter=",",
        header="wealth,rank,probability,c1,c2,weighted_h1,weighted_h2,state_welfare",
        comments="", fmt="%.17g",
    )

packages = {}
for name in ["numpy", "scipy", "matplotlib"]:
    try:
        packages[name] = version(name)
    except PackageNotFoundError:
        packages[name] = "NOT INSTALLED"
environment = {"python": sys.version, "platform": platform.platform(), "packages": packages}
environment["matches_requested_python_3_12"] = sys.version_info[:2] == (3, 12)
config = {"status": "PROVISIONAL: awaiting Parsa", "m": number_of_states, "n": number_of_agents,
          "wealth": wealth.tolist(), "ranks": ranks.tolist(), "probabilities": probabilities.tolist(),
          "pareto_weights": pareto_weights.tolist(), "agent_types": ["h1", "h2"],
          "quadrature": "uniform midpoint", "tolerance": TOLERANCE}
(OUT / "config.json").write_text(json.dumps(config, indent=2))
(OUT / "results.json").write_text(json.dumps(results, indent=2))
(OUT / "environment.json").write_text(json.dumps(environment, indent=2))
(OUT / "requirements-recorded.txt").write_text("\n".join(
    name + "==" + value for name, value in packages.items() if value != "NOT INSTALLED") + "\n")
(OUT / "review_status.json").write_text(json.dumps({
    "computational_checks": "passed",
    "Parsa_model_card_confirmation": "pending",
    "Parsa_independent_objective_comparison": "pending",
    "supplied_baseline": "not run: separate code package not supplied",
    "day1_joint_gate": "pending",
}, indent=2))

export_names = ["equal.csv", "one_third_two_thirds.csv", "config.json", "results.json",
                "environment.json", "requirements-recorded.txt", "review_status.json"]
with zipfile.ZipFile("day1_results.zip", "w", zipfile.ZIP_DEFLATED) as archive:
    for name in export_names:
        archive.write(OUT / name, arcname="risk_sharing_day1/" + name)
print("Saved: risk_sharing_day1/ and day1_results.zip")

Saved: risk_sharing_day1/ and day1_results.zip


**Finish line:** computational checks pass; Parsa confirms the example and independently verifies both welfare values. Until then, Day 1 is ready for review, with team sign-off pending. More detail is in `day01/README.md` in the repository.
